# CycloSafe — Objective 2: Cyclone Intensity Estimation V3 (Balanced 30k)
## Deep Learning Regression with EfficientNetB0 on 30,000 Balanced Satellite Images

This notebook trains a balanced continuous intensity estimation model ($V_{\max}$ in knots) solving the "regression collapse" issue.

**Dataset Configuration (30,000 Perfectly Balanced Images):**
- **Depression (20–34 kt):** 7,500 images
- **Tropical Storm (35–64 kt):** 7,500 images
- **Severe Cyclone (65–89 kt):** 7,500 images
- **Very Severe / Violent (90–185 kt):** 7,500 images

**Target Output:**
- Saves `best_cyclone_intensity_v3.keras` directly to your Google Drive root.


## Step 1: Verify Hardware Accelerator (GPU)


In [ ]:
import tensorflow as tf
import os, sys, time

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow Version:', tf.__version__)
if gpus:
    print('✅ GPU Detected:', gpus)
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    print('⚠️ WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> Select T4 GPU.')


## Step 2: Mount Google Drive & Extract Balanced 30k Dataset


In [ ]:
from google.colab import drive
import zipfile, shutil

drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/cyclosafe_intensity_balanced_30k.zip'
EXTRACT_DIR = '/content/dataset'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"Could not find {ZIP_PATH}! Please upload 'cyclosafe_intensity_balanced_30k.zip' to your Google Drive root.")

if not os.path.exists(EXTRACT_DIR):
    print(f"Extracting {ZIP_PATH} to high-speed NVMe SSD...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("✅ Extraction complete!")
else:
    print(f"Dataset already extracted at {EXTRACT_DIR}")

# Verify contents
images_dir = os.path.join(EXTRACT_DIR, 'images')
labels_csv = os.path.join(EXTRACT_DIR, 'metadata', 'labels.csv')
num_images = len([f for f in os.listdir(images_dir) if f.endswith('.jpg')])
print(f"Verified dataset images: {num_images:,}")
print(f"Labels CSV present: {os.path.exists(labels_csv)}")


## Step 3: Load Metadata & Group-Split by Storm ID (Zero Leakage)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt

df = pd.read_csv(labels_csv)
print(f"Total metadata records: {len(df):,}")
print("Intensity category breakdown:")
print(df['intensity_category'].value_counts())

df['full_image_path'] = df['image_path'].apply(lambda p: os.path.join(EXTRACT_DIR, p))
df = df[df['full_image_path'].apply(os.path.exists)].copy()
df['wind_speed_knots'] = df['wind_speed_knots'].astype(float)

# Strict group split: storms in Train NEVER appear in Validation or Test
gss = GroupShuffleSplit(n_splits=1, train_size=0.85, random_state=42)
train_idx, temp_idx = next(gss.split(df, groups=df['storm_id']))

train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

gss_val = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df['storm_id']))

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

print(f"\nDataset Partitioning:")
print(f"  - Training Set:   {len(train_df):,} images ({train_df['storm_id'].nunique()} storms)")
print(f"  - Validation Set: {len(val_df):,} images ({val_df['storm_id'].nunique()} storms)")
print(f"  - Test Set:       {len(test_df):,} images ({test_df['storm_id'].nunique()} storms)")


## Step 4: High-Performance tf.data Streaming Pipeline


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def parse_image(path, label):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0  # Normalize to [0, 1]
    return img, tf.cast(label, tf.float32)

def augment_image(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.06)
    img = tf.image.random_contrast(img, lower=0.94, upper=1.06)
    return img, label

def create_dataset(dataframe, is_training=False):
    paths = dataframe['full_image_path'].values
    labels = dataframe['wind_speed_knots'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        ds = ds.shuffle(buffer_size=4096)
        ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(train_df, is_training=True)
val_ds = create_dataset(val_df, is_training=False)
test_ds = create_dataset(test_df, is_training=False)
print("✅ High-throughput data pipelines initialized.")


## Step 5: Build EfficientNetB0 Regression Model


In [ ]:
from tensorflow.keras import layers, models, optimizers, callbacks, losses

def build_cyclone_intensity_model():
    base_model = tf.keras.applications.EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
    )
    base_model.trainable = False  # Warmup head first
    
    inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="satellite_input")
    x = base_model(inputs)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(256, activation="relu", name="dense_256")(x)
    x = layers.Dropout(0.3, name="dropout_0.3")(x)
    x = layers.Dense(64, activation="relu", name="dense_64")(x)
    outputs = layers.Dense(1, activation="linear", name="wind_speed_knots")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="EfficientNetB0_Intensity_V3")
    return model, base_model

model, base_model = build_cyclone_intensity_model()
model.summary()

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss=losses.Huber(delta=12.0),
    metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
)


## Step 6: Phase 1 — Warmup Head Training (3 Epochs)


In [ ]:
MODEL_SAVE_PATH = '/content/drive/MyDrive/best_cyclone_intensity_v3.keras'

cb_phase1 = [
    callbacks.ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_mae', mode='min', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor='val_mae', patience=2, restore_best_weights=True)
]

print("Starting Phase 1: Warmup regression head...")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    callbacks=cb_phase1
)


## Step 7: Phase 2 — End-to-End Fine-Tuning (Top 60 Layers Unfrozen)


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-60]:
    layer.trainable = False

print(f"Unfrozen top 60 layers of EfficientNet for deep meteorological learning.")

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss=losses.Huber(delta=12.0),
    metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
)

cb_phase2 = [
    callbacks.ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_mae', mode='min', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor='val_mae', patience=4, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

print("Starting Phase 2: Deep fine-tuning (10 epochs)...")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=cb_phase2
)


## Step 8: Comprehensive Test Set Evaluation Across All Categories


In [ ]:
print("Evaluating on held-out test storms...")
eval_results = model.evaluate(test_ds, verbose=1)
print(f"\n🎯 Final Test Set Performance:")
print(f"  - Test MAE:  {eval_results[1]:.2f} knots")
print(f"  - Test RMSE: {eval_results[2]:.2f} knots")

# Predict on test set
preds = model.predict(test_ds, verbose=0).flatten()
actuals = test_df['wind_speed_knots'].values[:len(preds)]

eval_df = pd.DataFrame({'actual': actuals, 'predicted': preds, 'category': test_df['intensity_category'].values[:len(preds)]})
print("\nCategory-wise Mean Absolute Error:")
for cat, grp in eval_df.groupby('category'):
    cat_mae = np.mean(np.abs(grp['actual'] - grp['predicted']))
    print(f"  - {cat:<22}: MAE = {cat_mae:.2f} knots (Sample: {len(grp):,})")

print(f"\n✅ Done! The trained model is saved at: {MODEL_SAVE_PATH}")
